In [1]:
import os
ea_dir = r"C:\Users\user\Downloads\GSE148812_clean"
print(os.listdir(ea_dir))

['checkpoint0_snp_clean.txt', 'checkpoint1_metadata_binary.csv', 'checkpoint2b_metadata_relatedness_filtered.csv', 'checkpoint2_metadata_sample_filtered.csv', 'checkpoint2_snp_sample_filtered.txt', 'checkpoint3_snp_probe_filtered.txt', 'checkpoint4_genotype_counts.csv', 'checkpoint5_hwe_results.csv', 'checkpoint6_snp_imputed.txt', 'checkpoint7b_snp_encoded_012_relatedness_filtered.csv', 'checkpoint7_snp_encoded_012.csv', 'checkpoint8_final_features_targets.csv', 'checkpoint9_doubleml_stability_results_cpd.csv', 'checkpoint9_doubleml_stability_results_relatedness_filtered.csv', 'confounders_X.npy', 'confounders_X_cpd.npy', 'confounders_X_relatedness_filtered.npy', 'gene_map_cpd_final_grch37.json', 'gene_map_smoking_sensitivity_grch37.json', 'pc1_gender_admixture_summary.csv', 'pca_diagnostics.csv', 'pca_diagnostics_relatedness_filtered.csv', 'pc_col_names_cpd_final.json', 'pc_col_names_smoking_sensitivity.json', 'pc_input_cpd_final.npy', 'pc_input_smoking_sensitivity.npy', 'probe_major_

In [2]:
import pandas as pd
import os

ea_dir = r"C:\Users\user\Downloads\GSE148812_clean"
df = pd.read_csv(os.path.join(ea_dir, "checkpoint7b_snp_encoded_012_relatedness_filtered.csv"), nrows=5)
print(df.columns[:5].tolist())
print(df['probe_id'].head(10).tolist() if 'probe_id' in df.columns else "no probe_id column - check first col name")

['probe_id', '1900001', '1900002', '1900003', '1900004']
['exm2268640-0_B_F_1984844585', 'exm41-0_B_F_1921435147', 'exm1916089-0_B_R_1927689775', 'exm44-0_B_R_1921538602', 'exm46-0_T_F_1921333919']


In [3]:
import pandas as pd
import numpy as np
import re
import os
import json
import gc

ea_dir = r"C:\Users\user\Downloads\GSE148812_clean"
aa_dir = r"C:\Users\user\Downloads\GSE148375_clean"
manifest_path = r"C:\Users\user\Downloads\HumanExome-12-v1-0-B.csv"

def strip_address_suffix(pid):
    return re.sub(r'_\d+$', '', pid)

# 1. Reuse the gene reference table from AA (genome build data, not cohort-specific)
genes_clean = pd.read_csv(os.path.join(aa_dir, "grch37_genes_clean.csv"))
print("Gene reference loaded:", genes_clean.shape)

# 2. Map EA's SNPs to genes (need full tested SNP list first - EA's DoubleML output)
manifest_df = pd.read_csv(manifest_path, skiprows=7, low_memory=False)
manifest_df["core_name"] = manifest_df["IlmnID"].map(strip_address_suffix)
pos_lookup = manifest_df.set_index("core_name")[["Chr", "MapInfo"]]

snp_list = pd.read_csv(os.path.join(ea_dir, "checkpoint9_doubleml_stability_results_relatedness_filtered.csv"))
snp_list["core_name"] = snp_list["probe_id"].map(strip_address_suffix)
snp_list = snp_list.merge(pos_lookup, left_on="core_name", right_index=True, how="left")
snp_list = snp_list.dropna(subset=["Chr", "MapInfo"])
print("EA SNPs with resolved positions:", len(snp_list))

snp_to_gene_ea = {}
for chrom in snp_list["Chr"].unique():
    chrom_str = str(chrom)
    genes_this_chrom = genes_clean[genes_clean["chrom"] == chrom_str].sort_values("start")
    snps_this_chrom = snp_list[snp_list["Chr"].astype(str) == chrom_str]
    if len(genes_this_chrom) == 0 or len(snps_this_chrom) == 0:
        continue
    starts = genes_this_chrom["start"].values
    ends = genes_this_chrom["end"].values
    names = genes_this_chrom["gene_name"].values
    for _, row in snps_this_chrom.iterrows():
        pos = row["MapInfo"]
        idx = np.searchsorted(starts, pos, side="right") - 1
        match = None
        for j in range(max(0, idx-3), min(len(starts), idx+4)):
            if starts[j] <= pos <= ends[j]:
                match = names[j]
                break
        snp_to_gene_ea[row["probe_id"]] = match if match else "intergenic"

print("EA SNPs mapped:", len(snp_to_gene_ea))
n_intergenic = sum(1 for g in snp_to_gene_ea.values() if g == "intergenic")
print("Intergenic:", n_intergenic, "| With gene:", len(snp_to_gene_ea) - n_intergenic)

with open(os.path.join(ea_dir, "snp_to_gene_map_full_EA.json"), "w") as f:
    json.dump(snp_to_gene_ea, f)
print("Saved EA SNP-to-gene map.")

Gene reference loaded: (57773, 5)
EA SNPs with resolved positions: 127416
EA SNPs mapped: 127416
Intergenic: 9499 | With gene: 117917
Saved EA SNP-to-gene map.


In [4]:
df_check = pd.read_csv(os.path.join(ea_dir, "checkpoint9_doubleml_stability_results_relatedness_filtered.csv"))
print(df_check.shape)
print((df_check['stability_fraction'] == 0).sum(), "of", len(df_check), "have stability_fraction == 0")

(127416, 3)
107572 of 127416 have stability_fraction == 0


In [5]:
import pandas as pd
import numpy as np
import json
import os
import gc

ea_dir = r"C:\Users\user\Downloads\GSE148812_clean"

# Load genotypes, metadata, gene map
encoded_df = pd.read_csv(os.path.join(ea_dir, "checkpoint7b_snp_encoded_012_relatedness_filtered.csv"))
meta_df = pd.read_csv(os.path.join(ea_dir, "checkpoint2b_metadata_relatedness_filtered.csv"))

with open(os.path.join(ea_dir, "snp_to_gene_map_full_EA.json")) as f:
    snp_to_gene_ea = json.load(f)

sample_cols = encoded_df.columns[1:].tolist()
print("EA genotype matrix:", encoded_df.shape)
print("EA metadata columns:", meta_df.columns.tolist())

EA genotype matrix: (239043, 1461)
EA metadata columns: ['sample_id', 'ethnicity', 'age', 'gender', 'cpd', 'hsi', 'ftnd', 'smoking_status', 'tissue']


In [6]:
# Align phenotype
meta_df["sample_id"] = meta_df["sample_id"].astype(str)
meta_df = meta_df.set_index("sample_id")
print(meta_df["smoking_status"].unique())  # confirm same categories as AA

meta_df["smoking_status_bin"] = (meta_df["smoking_status"] == "Smoker").astype(int)
pheno_ea = meta_df.loc[sample_cols, "smoking_status_bin"].astype(float)
print("EA phenotype aligned:", pheno_ea.shape)
print(pheno_ea.value_counts())

# Filter to genic SNPs
snp_gene_series_ea = pd.Series(snp_to_gene_ea)
snp_gene_series_ea = snp_gene_series_ea[snp_gene_series_ea != "intergenic"]

encoded_df_genic_ea = encoded_df[encoded_df["probe_id"].isin(snp_gene_series_ea.index)].copy()
encoded_df_genic_ea["gene"] = encoded_df_genic_ea["probe_id"].map(snp_gene_series_ea)
print("EA genic SNPs:", len(encoded_df_genic_ea))

del encoded_df
gc.collect()

# Compute direction and flip
geno_matrix_ea = encoded_df_genic_ea[sample_cols].values
pheno_vals_ea = pheno_ea.values

geno_centered_ea = geno_matrix_ea - geno_matrix_ea.mean(axis=1, keepdims=True)
pheno_centered_ea = pheno_vals_ea - pheno_vals_ea.mean()

numerator_ea = geno_centered_ea @ pheno_centered_ea
denom_ea = np.sqrt((geno_centered_ea**2).sum(axis=1) * (pheno_centered_ea**2).sum())
denom_ea[denom_ea == 0] = np.nan

corr_ea = numerator_ea / denom_ea
direction_ea = np.sign(np.nan_to_num(corr_ea, nan=0.0))
direction_ea[direction_ea == 0] = 1

print("EA SNPs flipped:", (direction_ea < 0).sum())
print("EA SNPs kept as-is:", (direction_ea >= 0).sum())

flipped_matrix_ea = np.where(direction_ea[:, None] < 0, 2 - geno_matrix_ea, geno_matrix_ea)
encoded_df_genic_ea[sample_cols] = flipped_matrix_ea

# Sum into gene-level signed burden
gene_burden_signed_ea = encoded_df_genic_ea.groupby("gene")[sample_cols].sum()
print("\nEA signed gene burden matrix shape:", gene_burden_signed_ea.shape)

gene_burden_signed_ea.to_csv(os.path.join(ea_dir, "gene_burden_matrix_signed_EA.csv"))
print("Saved.")

# Filter to protein-coding
protein_coding_genes = set(genes_clean[genes_clean["Gene type"] == "protein_coding"]["gene_name"])
gene_burden_ea_pc = gene_burden_signed_ea[gene_burden_signed_ea.index.isin(protein_coding_genes)]
print("EA protein-coding filtered:", gene_burden_ea_pc.shape)

gene_burden_ea_pc.to_csv(os.path.join(ea_dir, "gene_burden_matrix_signed_protein_coding_EA.csv"))
print("Saved final EA gene burden matrix.")

<ArrowStringArray>
['Smoker', 'Non-smoker']
Length: 2, dtype: str
EA phenotype aligned: (1460,)
smoking_status_bin
1.0    795
0.0    665
Name: count, dtype: int64
EA genic SNPs: 117917
EA SNPs flipped: 59508
EA SNPs kept as-is: 58409

EA signed gene burden matrix shape: (16869, 1460)
Saved.
EA protein-coding filtered: (15128, 1460)
Saved final EA gene burden matrix.
